# Exercice 4 - Regression on the provided dataset

Donnees: `FTML/project/data/regression/`.

Objectif: comparer au moins deux methodes de regression sans utiliser le test set pendant la selection de modele. Le score final est un $R^2$ sur test.

In [1]:
import numpy as np
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR

def find_existing_path(*candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Aucun chemin de donnees valide trouve')

DATA = find_existing_path(
    'FTML/project/data/regression',
    '../../FTML/project/data/regression',
    '../FTML/project/data/regression',
)
X_train = np.load(DATA / 'X_train.npy')
y_train = np.load(DATA / 'y_train.npy').ravel()
X_test = np.load(DATA / 'X_test.npy')
y_test = np.load(DATA / 'y_test.npy').ravel()

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (200, 200)
X_test: (200, 200)
y_train: (200,)
y_test: (200,)


Le dataset contient autant de variables que d'observations d'entrainement (`n=d=200`). Une regression lineaire non regularisee risque donc de sur-apprendre. Les methodes choisies sont Ridge, Lasso, ElasticNet et SVR RBF. La selection d'hyperparametres se fait par validation croisee 5-fold sur le train set uniquement.

In [2]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Ridge': (
        make_pipeline(StandardScaler(), Ridge()),
        {'ridge__alpha': np.logspace(-4, 4, 17)},
    ),
    'Lasso': (
        make_pipeline(StandardScaler(), Lasso(max_iter=50_000)),
        {'lasso__alpha': np.logspace(-5, 1, 13)},
    ),
    'ElasticNet': (
        make_pipeline(StandardScaler(), ElasticNet(max_iter=50_000)),
        {
            'elasticnet__alpha': np.logspace(-5, 1, 13),
            'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
        },
    ),
    'SVR_RBF': (
        make_pipeline(StandardScaler(), SVR(kernel='rbf')),
        {
            'svr__C': [1, 3, 10, 30, 100],
            'svr__gamma': ['scale', 0.001, 0.003, 0.01, 0.03],
            'svr__epsilon': [0.01, 0.1, 0.3],
        },
    ),
}

results = []
for name, (estimator, grid) in models.items():
    search = GridSearchCV(estimator, grid, cv=cv, scoring='r2', n_jobs=-1)
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)
    results.append({
        'model': name,
        'best_cv_r2': search.best_score_,
        'test_r2': r2_score(y_test, y_pred),
        'test_rmse': mean_squared_error(y_test, y_pred) ** 0.5,
        'params': search.best_params_,
    })

for row in sorted(results, key=lambda r: r['test_r2'], reverse=True):
    print(row['model'])
    print('  CV R2:   ', round(row['best_cv_r2'], 4))
    print('  Test R2: ', round(row['test_r2'], 4))
    print('  RMSE:    ', round(row['test_rmse'], 4))
    print('  params:  ', row['params'])

ElasticNet
  CV R2:    0.9321
  Test R2:  0.9208
  RMSE:     0.2415
  params:   {'elasticnet__alpha': np.float64(0.03162277660168379), 'elasticnet__l1_ratio': 0.9}
Lasso
  CV R2:    0.9314
  Test R2:  0.9203
  RMSE:     0.2422
  params:   {'lasso__alpha': np.float64(0.03162277660168379)}
Ridge
  CV R2:    0.6228
  Test R2:  0.7153
  RMSE:     0.4578
  params:   {'ridge__alpha': np.float64(10.0)}
SVR_RBF
  CV R2:    0.5947
  Test R2:  0.6343
  RMSE:     0.5189
  params:   {'svr__C': 10, 'svr__epsilon': 0.01, 'svr__gamma': 0.001}


Resultats obtenus localement:

| Modele | CV R2 | Test R2 | Commentaire |
|---|---:|---:|---|
| ElasticNet | 0.9321 | 0.9208 | Meilleur score test, regularisation surtout L1. |
| Lasso | 0.9314 | 0.9203 | Presque identique a ElasticNet. |
| Ridge | 0.6228 | 0.7153 | Regularisation L2 seule insuffisante ici. |
| SVR RBF | 0.5947 | 0.6343 | Moins adapte avec ces hyperparametres et peu d'observations. |

Conclusion: Lasso et ElasticNet sont les meilleurs choix. Le score test depasse l'objectif demande de `R2 > 0.88`.